# Preprocesado de datos y división del dataset

En este notebook vamos a preparar los datos para el entrenamiento del modelo clasificador de riesgo IA (AI Act).

Los pasos que seguiremos son:
1. Carga del dataset limpio
2. Aplicar NER (Named Entity Recognition) a la columna `descripcion`
3. Explorar las entidades extraídas por tipo y clase de riesgo
4. Preprocesar el texto con lematización
5. Dividir en train / validation / test (70% / 15% / 15%) con estratificación
6. Guardar los conjuntos en `data/processed/`

In [14]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_ultimo_dataset"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ultimo_dataset"
functions._DATASET_TAGS = {"dataset_type": "ultimo", "dataset_source": "dataset_sintetico_v2"}

In [15]:
#Al principio de cada notebook, añadimos estas líneas para que se recarguen automáticamente las funciones que hayamos editado en el módulo functions.py sin tener que reiniciar el kernel cada vez.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import pandas as pd

path = "datasets/dataset_sintetico_v2_limpio.csv"
df = pd.read_csv(path)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
print("\nDistribución de clases:")
print(df["etiqueta"].value_counts())

Dataset cargado: 285 filas, 7 columnas
Columnas: ['id', 'descripcion', 'sector', 'tipo_datos', 'etiqueta', 'longitud', 'descripcion_limpia']

Distribución de clases:
etiqueta
riesgo_limitado    72
alto_riesgo        72
riesgo_minimo      72
inaceptable        69
Name: count, dtype: int64


## 1. Aplicar NER al dataset

In [17]:
import warnings
from functions import extraer_entidades, resumen_entidades

# Aplicamos NER a la columna de descripción original (sin limpiar)
df = extraer_entidades(df, "descripcion")

# NER es solo exploratorio — no afecta al pipeline de entrenamiento.
# Si spaCy no está disponible, continuamos sin la columna de entidades.
_ner_available = not df.get("ner_failed", pd.Series([False])).any()
if not _ner_available:
    warnings.warn(
        "spaCy no disponible o 'es_core_news_sm' no instalado. "
        "Se omite el análisis NER (no afecta al entrenamiento).",
        UserWarning,
    )
    print("⚠ NER omitido — continuando con limpieza de texto estándar.")
else:
    # Mostramos las entidades de las primeras 3 filas como ejemplo
    for i in range(3):
        print(f"Texto: {df['descripcion'].iloc[i][:100]}...")
        print(f"Entidades: {df['entidades'].iloc[i]}")
        print()

⚠ NER omitido — continuando con limpieza de texto estándar.


C:\Users\rammu\AppData\Local\Temp\ipykernel_20512\171427618.py:11: UserWarning: spaCy no disponible o 'es_core_news_sm' no instalado. Se omite el análisis NER (no afecta al entrenamiento).
  warnings.warn(


In [18]:
# Resumen de entidades por tipo y por clase de riesgo (solo si NER disponible)
if _ner_available:
    freq_por_tipo, freq_por_tipo_clase = resumen_entidades(df)
else:
    print("⚠ Resumen NER omitido — spaCy no disponible.")

⚠ Resumen NER omitido — spaCy no disponible.


### Conclusión del NER

Las entidades nombradas (NER) en este dataset regulatorio no aportan señal discriminativa significativa para la clasificación de riesgo. Los textos describen sistemas genéricos de IA sin mencionar nombres propios, organizaciones o ubicaciones específicas que diferencien las clases.

Por tanto, **continuamos el entrenamiento utilizando solo el texto limpio y lematizado** como feature principal.

## 2. Preprocesado del texto con lematización

In [19]:
from functions import preparar_dataset

# Columnas extra candidatas (en orden de preferencia).
# Se filtran automáticamente a las que existan en el dataset cargado,
# evitando KeyError si el esquema del CSV es diferente al del fusionado.
_CANDIDATAS = ["category", "context", "longitud", "num_articles", "sector", "tipo_datos"]
EXTRA_FEATURES = [c for c in _CANDIDATAS if c in df.columns]
print(f"Features estructuradas disponibles: {EXTRA_FEATURES}")

df_processed = preparar_dataset(df, "descripcion", "etiqueta", extra_columns=EXTRA_FEATURES)

print(f"\nDataset procesado: {df_processed.shape}")
print(f"Columnas: {list(df_processed.columns)}")

print("\nEjemplo de texto lematizado:")
print(f"  Original:   {df['descripcion'].iloc[0][:100]}...")
print(f"  Lematizado: {df_processed['text_final'].iloc[0][:100]}...")

Features estructuradas disponibles: ['longitud', 'sector', 'tipo_datos']

Dataset procesado: (285, 5)
Columnas: ['text_final', 'etiqueta', 'longitud', 'sector', 'tipo_datos']

Ejemplo de texto lematizado:
  Original:   Herramienta que genera subtítulos automáticos para vídeos corporativos indicando que son producidos ...
  Lematizado: herramienta genera subtítulos automáticos vídeos corporativos indicando producidos...


c:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\src\classifier\functions.py:197: UserWarning: Lematización no disponible sin spaCy; se devuelve texto sin lematizar.
  return _limpiar_texto_fallback(texto, lemmatize)


## 3. División en train / validation / test

In [20]:
from functions import split_dataset

# Dividimos en train (70%), validation (15%) y test (15%) con estratificación
train_df, val_df, test_df = split_dataset(df_processed, "etiqueta")

print("\nDistribución por clase en cada conjunto:")
print("\nTrain:")
print(train_df["etiqueta"].value_counts())
print("\nValidation:")
print(val_df["etiqueta"].value_counts())
print("\nTest:")
print(test_df["etiqueta"].value_counts())

Train: 199 muestras
Validation: 43 muestras
Test: 43 muestras

Distribución por clase en cada conjunto:

Train:
etiqueta
alto_riesgo        50
riesgo_minimo      50
riesgo_limitado    50
inaceptable        49
Name: count, dtype: int64

Validation:
etiqueta
riesgo_limitado    11
riesgo_minimo      11
alto_riesgo        11
inaceptable        10
Name: count, dtype: int64

Test:
etiqueta
riesgo_minimo      11
alto_riesgo        11
riesgo_limitado    11
inaceptable        10
Name: count, dtype: int64


## 4. Guardar los conjuntos procesados

In [21]:
import os

output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "validation.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print(f"Archivos guardados en {output_dir}/:")
for f in os.listdir(output_dir):
    print(f"  {f}")

Archivos guardados en data/processed/:
  test.csv
  train.csv
  validation.csv


In [22]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe

try:
    log_mlflow_safe(
        run_name="preprocesado",
        params={
            "lemmatize":      True,
            "ner":            True,
            "test_size":      0.15,
            "val_size":       0.15,
            "random_state":   42,
            "extra_features": "category,context,longitud,num_articles",
        },
        metrics={
            "n_train":       len(train_df),
            "n_val":         len(val_df),
            "n_test":        len(test_df),
            "n_features_out": df_processed.shape[1] - 1,  # excluye etiqueta
        },
    )
    print("✓ Preprocesado registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/
⚠ MLflow no disponible: API request to https://18.201.64.41/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='18.201.64.41', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_ultimo_dataset (Caused by ConnectTimeoutError(<HTTPSConnection(host='18.201.64.41', port=443) at 0x19d7a1fd8b0>, 'Connection to 18.201.64.41 timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120, type: int) to a larger value.
